# 03 - LLM Judge and Semantic Reranking

This notebook demonstrates the online semantic reranker on three examples and retains the offline blind model-tournament workflow.

In [2]:
%cd /content/LLM-Project-SemEval-Humor-Generation
%pip install -r requirements-colab.txt

import os
from getpass import getpass

if not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
    token = getpass("Hugging Face token for Llama (hidden; leave blank to use another model): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token

/content/LLM-Project-SemEval-Humor-Generation


from pathlib import Path
import json
import os
import subprocess
import sys

import torch

# Monta Google Drive se siamo in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

RUN_RERANK_TEST = True
TEST_MODEL = "llama"  # Change to "qwen" or "mistral" if preferred.
TEST_EXAMPLES = 3
TEST_CANDIDATES = 5
TEST_N_DOCS = 25_000
TEST_K = 4

root = Path.cwd().resolve()
if not (root / "scripts/run_rag_generate.py").exists():
    raise RuntimeError("Run the Colab setup cell first.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running real inference.")
if TEST_MODEL == "llama" and not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
    raise RuntimeError("Set HF_TOKEN above or change TEST_MODEL to qwen/mistral.")

# Imposta il percorso di salvataggio su Google Drive
drive_rag_dir = Path("/content/drive/MyDrive/semeval_humor_outputs/rag")

# Fallback in caso venga eseguito in locale e non su Colab
if not Path("/content").exists():
     drive_rag_dir = root / "data/generated/rag"

drive_rag_dir.mkdir(parents=True, exist_ok=True)

test_output = drive_rag_dir / (
    f"{TEST_MODEL}_rag_{TEST_N_DOCS // 1000}k_k{TEST_K}_"
    f"bestof{TEST_CANDIDATES}_llmjudge_test3.jsonl"
)

# Forza l'abilitazione del modulo di selezione (Best-of-N + Giudice LLM) nel RAG Config temporaneamente
# In modo da sovrascrivere l'impostazione "selection.enabled: false"
rag_config_path = root / "configs/rag.yaml"
temp_rag_config_path = root / "configs/temp_rag_enabled.yaml"

import yaml
with rag_config_path.open() as f:
    rag_cfg = yaml.safe_load(f)
    
if "selection" not in rag_cfg:
    rag_cfg["selection"] = {}
rag_cfg["selection"]["enabled"] = True
rag_cfg["selection"]["num_candidates"] = TEST_CANDIDATES

with temp_rag_config_path.open("w") as f:
    yaml.dump(rag_cfg, f)


if RUN_RERANK_TEST:
    command = [
        sys.executable,
        str(root / "scripts/run_rag_generate.py"),
        "--model", TEST_MODEL,
        "--input", str(root / "data/raw/task-a-en.tsv"),
        "--output", str(test_output),
        "--rag-config", str(temp_rag_config_path), # Uso la configurazione temporanea modificata
        "--n-docs", str(TEST_N_DOCS),
        "--k", str(TEST_K),
        "--apply-to", "headline",
        "--limit", str(TEST_EXAMPLES),
        "--overwrite",
        "--verbose",
    ]
    print("Generating 15 sampled jokes and running semantic reranking...")
    subprocess.run(command, check=True, cwd=root)
else:
    print("Set RUN_RERANK_TEST = True to run inference.")

if not test_output.exists():
    raise RuntimeError(f"Expected output not found: {test_output}")

with test_output.open(encoding="utf-8") as file:
    rerank_test_rows = [json.loads(line) for line in file if line.strip()]

if len(rerank_test_rows) != TEST_EXAMPLES:
    raise RuntimeError(f"Expected {TEST_EXAMPLES} records, found {len(rerank_test_rows)}.")
print(f"Saved {len(rerank_test_rows)} reranked records to {test_output}")


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import torch

# Monta Google Drive se siamo in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

RUN_RERANK_TEST = True
TEST_MODEL = "llama"  # Change to "qwen" or "mistral" if preferred.
TEST_EXAMPLES = 3
TEST_CANDIDATES = 5
TEST_N_DOCS = 25_000
TEST_K = 5

root = Path.cwd().resolve()
if not (root / "scripts/run_rag_generate.py").exists():
    raise RuntimeError("Run the Colab setup cell first.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running real inference.")
if TEST_MODEL == "llama" and not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
    raise RuntimeError("Set HF_TOKEN above or change TEST_MODEL to qwen/mistral.")

# Imposta il percorso di salvataggio su Google Drive
drive_rag_dir = Path("/content/drive/MyDrive/semeval_humor_outputs/rag")

# Fallback in caso venga eseguito in locale e non su Colab
if not Path("/content").exists():
     drive_rag_dir = root / "data/generated/rag"

drive_rag_dir.mkdir(parents=True, exist_ok=True)

test_output = drive_rag_dir / (
    f"{TEST_MODEL}_rag_{TEST_N_DOCS // 1000}k_k{TEST_K}_"
    f"bestof{TEST_CANDIDATES}_llmjudge_test3.jsonl"
)

if RUN_RERANK_TEST:
    command = [
        sys.executable,
        str(root / "scripts/run_rag_generate.py"),
        "--model", TEST_MODEL,
        "--input", str(root / "data/raw/task-a-en.tsv"),
        "--output", str(test_output),
        "--rag-config", str(root / "configs/rag.yaml"),
        "--n-docs", str(TEST_N_DOCS),
        "--k", str(TEST_K),
        "--apply-to", "headline",
        "--limit", str(TEST_EXAMPLES),
        "--overwrite",
        "--verbose",
    ]
    print("Generating 15 sampled jokes and running semantic reranking...")
    subprocess.run(command, check=True, cwd=root)
else:
    print("Set RUN_RERANK_TEST = True to run inference.")

if not test_output.exists():
    raise RuntimeError(f"Expected output not found: {test_output}")

with test_output.open(encoding="utf-8") as file:
    rerank_test_rows = [json.loads(line) for line in file if line.strip()]

if len(rerank_test_rows) != TEST_EXAMPLES:
    raise RuntimeError(f"Expected {TEST_EXAMPLES} records, found {len(rerank_test_rows)}.")
print(f"Saved {len(rerank_test_rows)} reranked records to {test_output}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating 15 sampled jokes and running semantic reranking...


CalledProcessError: Command '['/usr/bin/python3', '/content/LLM-Project-SemEval-Humor-Generation/scripts/run_rag_generate.py', '--model', 'llama', '--input', '/content/LLM-Project-SemEval-Humor-Generation/data/raw/task-a-en.tsv', '--output', '/content/drive/MyDrive/semeval_humor_outputs/rag/llama_rag_25k_k5_bestof5_llmjudge_test3.jsonl', '--rag-config', '/content/LLM-Project-SemEval-Humor-Generation/configs/rag.yaml', '--n-docs', '25000', '--k', '5', '--apply-to', 'headline', '--limit', '3', '--overwrite', '--verbose']' returned non-zero exit status 1.

In [17]:
import pandas as pd
from IPython.display import display

candidate_table = []
summary_table = []
fallback_table = []

for example_number, row in enumerate(rerank_test_rows, start=1):
    metadata = row.get("metadata", {})
    reranker = metadata.get("reranker", {})
    all_candidates = metadata.get("candidates", [])
    sampled_candidates = [candidate for candidate in all_candidates if not candidate.get("fallback")]
    fallback_candidates = [candidate for candidate in all_candidates if candidate.get("fallback")]
    if len(sampled_candidates) != TEST_CANDIDATES:
        raise RuntimeError(
            f"Example {example_number}: expected {TEST_CANDIDATES} sampled candidates, "
            f"found {len(sampled_candidates)}."
        )

    valid_sample_indexes = [
        index for index, candidate in enumerate(sampled_candidates) if candidate.get("valid")
    ]
    winner_among_valid = reranker.get("winner_index_among_valid")
    selected_sample_index = None
    if winner_among_valid is not None and winner_among_valid < len(valid_sample_indexes):
        selected_sample_index = valid_sample_indexes[winner_among_valid]
    if selected_sample_index is None and not metadata.get("fallback_used"):
        selected_sample_index = next(
            (
                index
                for index, candidate in enumerate(sampled_candidates)
                if candidate.get("valid") and candidate.get("generated_joke") == row.get("generated_joke")
            ),
            None,
        )

    for candidate_index, candidate in enumerate(sampled_candidates):
        candidate_table.append({
            "example": example_number,
            "id": row.get("id"),
            "candidate": candidate_index,
            "valid": candidate.get("valid"),
            "llm_reward": candidate.get("score"),
            "selected": candidate_index == selected_sample_index,
            "errors": ", ".join(candidate.get("constraint_errors", [])),
            "joke": candidate.get("generated_joke"),
        })

    for candidate in fallback_candidates:
        fallback_table.append({
            "example": example_number,
            "valid": candidate.get("valid"),
            "errors": ", ".join(candidate.get("constraint_errors", [])),
            "joke": candidate.get("generated_joke"),
        })

    summary_table.append({
        "example": example_number,
        "id": row.get("id"),
        "input": row.get("input"),
        "valid_candidates": len(valid_sample_indexes),
        "judge": reranker.get("scorer_type"),
        "judge_reason": reranker.get("reasoning"),
        "fallback_used": metadata.get("fallback_used"),
        "winner": row.get("generated_joke"),
    })

candidate_df = pd.DataFrame(candidate_table)
if len(candidate_df) != TEST_EXAMPLES * TEST_CANDIDATES:
    raise RuntimeError(f"Expected 15 sampled rows, found {len(candidate_df)}.")

pd.set_option("display.max_colwidth", 160)
print("15 SAMPLED CANDIDATES")
display(candidate_df)
print("RERANKING SUMMARY")
display(pd.DataFrame(summary_table))
if fallback_table:
    print("GREEDY FALLBACKS (not included in the 15 sampled candidates)")
    display(pd.DataFrame(fallback_table))

RuntimeError: Example 1: expected 5 sampled candidates, found 0.

## Optional offline blind tournament

This is separate from online reranking: it compares final outputs produced by different models.

In [ ]:
DATA = "data/raw/sample_task_a.tsv"
MOCK = "--mock"
!python scripts/run_judge.py --input-dir data/generated/baseline --output data/judged/base_judgments.jsonl --method base --dataset {DATA} {MOCK} --overwrite